In [11]:
import os
import json
from google import genai
from google.genai import types
import pandas as pd

# ==============================================================================
# CONFIGURACIÓN DE RUTAS Y ARCHIVOS (COMPATIBLE CON JUPYTER)
# ==============================================================================
CARPETA_ACTUAL = os.getcwd()
NOMBRE_CSV = "tabla_maestra_15_recetas_completa.csv"
RUTA_CSV = os.path.join(CARPETA_ACTUAL, NOMBRE_CSV)

def asegurar_entorno():
    """Crea el CSV maestro básico si no existe en la carpeta."""
    if not os.path.exists(RUTA_CSV):
        print(f"⚠️ Creando archivo CSV maestro en: {RUTA_CSV}")
        df_base = pd.DataFrame(columns=[
            "ID", "Perfume Inspiración", "Marca / Estilo", "Precio Original (€)", 
            "Tamaño Original (ml)", "Precio Original (€/ml)", "FIJ (ml)", "BER (ml)", 
            "TAM (ml)", "CNP (ml)", "TON (ml)", "VAI (ml)", "YLA (ml)", "AMB (ml)", 
            "MUS (ml)", "Volumen Esencias (ml)", "ALC (ml)", "Coste Esencias (€)", 
            "Coste Fijador (€)", "Coste Alcohol (€)", "Coste Total 100ml (€)", "Similitud Estimada"
        ])
        df_base.to_csv(RUTA_CSV, index=False, encoding='utf-8-sig')

def obtener_cliente_ia():
    """Inicializa el cliente de Gemini leyendo la variable de entorno."""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("🔑 [ERROR]: Configura tu variable GEMINI_API_KEY antes de ejecutar.")
    return genai.Client(api_key=api_key)

def guardar_receta_en_csv(nombre_perfume, marca, precio_orig, tam_orig, formula):
    asegurar_entorno()
    prices = {
        'FIJ': 5.68 / 50, 'ALC': 6.20 / 1000, 'BER': 5.84 / 50, 'TAM': 6.20 / 50,
        'CNP': 7.67 / 50, 'TON': 7.92 / 50, 'VAI': 4.21 / 50, 'YLA': 6.68 / 50,
        'AMB': 5.84 / 50, 'MUS': 5.38 / 50,
    }
    
    coste_esencias = sum(formula.get(k, 0.0) * prices.get(k, 0.0) for k in ['BER', 'TAM', 'CNP', 'TON', 'VAI', 'YLA', 'AMB', 'MUS'])
    coste_fijador = 1.0 * prices['FIJ']
    coste_alcohol = 69.0 * prices['ALC']
    coste_total = coste_esencias + coste_fijador + coste_alcohol
    precio_orig_ml = round(precio_orig / tam_orig, 2) if tam_orig > 0 else 0.0
    
    df = pd.read_csv(RUTA_CSV)
    nuevo_id = int(df['ID'].max() + 1) if 'ID' in df.columns and not df.empty else 1
    
    nueva_fila = {
        "ID": nuevo_id, "Perfume Inspiración": nombre_perfume, "Marca / Estilo": marca,
        "Precio Original (€)": precio_orig, "Tamaño Original (ml)": tam_orig, "Precio Original (€/ml)": precio_orig_ml,
        "FIJ (ml)": 1.0, "BER (ml)": formula.get('BER', 0.0), "TAM (ml)": formula.get('TAM', 0.0),
        "CNP (ml)": formula.get('CNP', 0.0), "TON (ml)": formula.get('TON', 0.0), "VAI (ml)": formula.get('VAI', 0.0),
        "YLA (ml)": formula.get('YLA', 0.0), "AMB (ml)": formula.get('AMB', 0.0), "MUS (ml)": formula.get('MUS', 0.0),
        "Volumen Esencias (ml)": sum(formula.values()), "ALC (ml)": 69.0,
        "Coste Esencias (€)": round(coste_esencias, 2), "Coste Fijador (€)": round(coste_fijador, 2),
        "Coste Alcohol (€)": round(coste_alcohol, 2), "Coste Total 100ml (€)": round(coste_total, 2),
        "Similitud Estimada": f"{formula.get('_similitud', 90)}%"
    }
    
    df_new = pd.concat([df, pd.DataFrame([nueva_fila])], ignore_index=True)
    df_new.to_csv(RUTA_CSV, index=False, encoding='utf-8-sig')
    print(f"💾 [AUTO-GUARDADO]: Receta añadida al CSV maestro (ID: {nuevo_id}).")

def clonar_perfume(nombre_perfume: str):
    client = obtener_cliente_ia()
    print(f"🔍 Analizando perfume objetivo: {nombre_perfume.upper()}...")

    prompt = f"""
    Eres un maestro perfumista y químico cosmético experto en ingeniería inversa.
    Analiza la pirámide olfativa oficial y el estilo del perfume comercial '{nombre_perfume}'.

    Devuelve un JSON estricto con los siguientes campos exactos:
    1. "marca_comercial": Nombre de la marca o casa del perfume.
    2. "precio_original_estimado": Precio medio de mercado en euros para botella grande.
    3. "tamano_original_ml": Tamaño estándar de venta en ml (ej: 100).
    4. "descripcion": Descripción breve (2-3 frases) del estilo y notas predominantes.
    5. "notas_detectadas": Lista con las principales notas oficiales.
    6. "formula_30ml": Fórmula adaptada en Extrait de Parfum (30% esencias) usando ÚNICAMENTE estas 8 esencias de Gran Velada: BER, TAM, CNP (Máximo 4.5 ml), TON, VAI, YLA, AMB, MUS. La suma exacta de las esencias debe ser 30.0 ml.
    7. "similitud_porcentaje": Porcentaje estimado de parecido (0 a 100).
    8. "receta_ideal_100ml": Diccionario con ingredientes reales ideales en ml (sumando 30ml) para clonarlo al 100%.
    9. "esencia_faltante": Esencia clave recomendada si la similitud < 90% (o "Ninguna").
    10. "precio_esencia_faltante_est": Precio estimado de esa esencia faltante en frasco de 50ml.

    Formato JSON exacto:
    {{
      "marca_comercial": "...", "precio_original_estimado": 120.0, "tamano_original_ml": 100,
      "descripcion": "...", "notas_detectadas": ["...", "..."],
      "formula_30ml": {{"BER": 0.0, "TAM": 0.0, "CNP": 0.0, "TON": 0.0, "VAI": 0.0, "YLA": 0.0, "AMB": 0.0, "MUS": 0.0}},
      "similitud_porcentaje": 88, "receta_ideal_100ml": {{"Ingrediente 1": 15.0}},
      "esencia_faltante": "...", "precio_esencia_faltante_est": 6.80
    }}
    """

    try:
        response = client.models.generate_content(
            model= "gemini-3.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                response_mime_type="application/json"
            ),
        )

        datos = json.loads(response.text)
        formula = datos["formula_30ml"]
        similitud = datos["similitud_porcentaje"]

        if formula.get("CNP", 0.0) > 4.5:
            formula["CNP"] = 4.5

        suma_esencias = sum(formula.values())
        if suma_esencias != 30.0 and suma_esencias > 0:
            diferencia = 30.0 - suma_esencias
            formula["VAI"] = round(formula.get("VAI", 0.0) + diferencia, 1)

        print(f"\n🏷️ MARCA: {datos['marca_comercial']} | ORIGINAL: {datos['precio_original_estimado']}€ / {datos['tamano_original_ml']}ml")
        print(f"📝 DESCRIPCIÓN: {datos['descripcion']}")
        print(f"📌 NOTAS: {', '.join(datos['notas_detectadas'])}")
        
        print(f"\n🧪 FÓRMULA ADAPTADA AL 30%:")
        print(f"   • Fijador (Dimeticona): 1.0 ml")
        nombres_largos = {
            'BER': 'Bergamota', 'TAM': 'Tamarindo', 'CNP': 'Canela/Naranja/Patchouli (Max 15%)',
            'TON': 'Haba Tonka', 'VAI': 'Vainilla', 'YLA': 'Ylang Ylang', 'AMB': 'Ámbar', 'MUS': 'Musk'
        }
        for sigla, ml in formula.items():
            if ml > 0:
                print(f"   • {nombres_largos.get(sigla, sigla)}: {ml:.1f} ml")
        print(f"   • Alcohol 96º: 69.0 ml")
        print(f"--------------------------------------------------------")
        print(f"📊 SIMILITUD ESTIMADA: {similitud}%")

        if similitud >= 90:
            print("🟢 ¡ADELANTE! Muy parecida con tus 8 esencias.")
        elif 80 <= similitud < 90:
            print("🟡 BASTANTE BIEN. Equivalencia muy disfrutable.")
        else:
            print(f"🔴 SIMILITUD BAJA. Te falta: {datos.get('esencia_faltante')} (~{datos.get('precio_esencia_faltante_est')}€).")

        print(f"\n🎯 RECETA IDEAL (Clon al 100%):")
        print(f"   • Fijador: 1.0 ml")
        for ing, ml in datos.get("receta_ideal_100ml", {}).items():
            print(f"   • {ing}: {ml:.1f} ml")
        print(f"   • Alcohol 96º: 69.0 ml")
        
        # Guardar automáticamente en el Jupyter Notebook
        formula['_similitud'] = similitud
        guardar_receta_en_csv(nombre_perfume.title(), datos['marca_comercial'], float(datos['precio_original_estimado']), int(datos['tamano_original_ml']), formula)

    except Exception as e:
        print(f"❌ Error: {e}")

print("Entorno Jupyter configurado y listo.")

Entorno Jupyter configurado y listo.


In [ ]:
# 1. Pega aquí tu clave de API para esta sesión (recuerda borrarla antes de subir a GitHub)
os.environ["GEMINI_API_KEY"] = "PEGA AQUÍ TU API KEY"

# 2. Escribe el perfume que quieres clonar y pulsa Play
clonar_perfume("")


🔍 Analizando perfume objetivo: ONE MILLION PACO RABANNE...

🏷️ MARCA: Paco Rabanne | ORIGINAL: 95.0€ / 100ml
📝 DESCRIPCIÓN: Una fragancia icónica de la familia olfativa amaderada especiada, que destaca por su estela ostentosa y vibrante. Combina acordes iniciales hespérides y mentolados con un corazón especiado y floral, asentándose sobre una base cálida, ambarina y de cuero dulce.
📌 NOTAS: Mandarina roja, Menta, Canela, Rosa, Acorde de cuero, Ámbar, Pachulí, Madera blanca

🧪 FÓRMULA ADAPTADA AL 30%:
   • Fijador (Dimeticona): 1.0 ml
   • Bergamota: 4.0 ml
   • Tamarindo: 4.0 ml
   • Canela/Naranja/Patchouli (Max 15%): 4.5 ml
   • Haba Tonka: 5.0 ml
   • Vainilla: 3.5 ml
   • Ylang Ylang: 2.0 ml
   • Ámbar: 4.0 ml
   • Musk: 3.0 ml
   • Alcohol 96º: 69.0 ml
--------------------------------------------------------
📊 SIMILITUD ESTIMADA: 87%
🟡 BASTANTE BIEN. Equivalencia muy disfrutable.

🎯 RECETA IDEAL (Clon al 100%):
   • Fijador: 1.0 ml
   • Alcohol Denat: 70.0 ml
   • Esencia 1 Millio